<a href="https://colab.research.google.com/github/Sruthi-Gudelli/Comment-Toxicity-Detection/blob/main/LSTM_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/train.csv')
train_data.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [ ]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [ ]:
train_data.shape

(159571, 8)

**Checking for nulls and duplicates**

In [ ]:
100*train_data.isnull().sum()/len(train_data)

,0
id,0.0
comment_text,0.0
toxic,0.0
severe_toxic,0.0
obscene,0.0
threat,0.0
insult,0.0
identity_hate,0.0


In [ ]:
train_data.duplicated().sum()

np.int64(0)

**Cleaning**

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

In [ ]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
def clean_text(text):
  text = re.sub(r'[^a-zA-Z]', ' ', text)
  text = text.lower()
  words = text.split()
  stop_words = stopwords.words('english')
  clean_words = [word for word in words if word not in stop_words]
  return " ".join(clean_words).strip()

train_data['cleaned_comments'] = train_data['comment_text'].apply(clean_text)

In [ ]:
train_data.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,cleaned_comments
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,explanation edits made username hardcore metal...
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,aww matches background colour seemingly stuck ...
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,hey man really trying edit war guy constantly ...
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,make real suggestions improvement wondered sec...
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,sir hero chance remember page


**Tokenization & Vectorization**





In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Build the Vocabulary (Tokenisation)
tokenizer = Tokenizer(num_words=30000, oov_token='<OOV>')
tokenizer.fit_on_texts(train_data['cleaned_comments'].values)

In [ ]:
sequences = tokenizer.texts_to_sequences(train_data['cleaned_comments'].values)

X = pad_sequences(sequences, maxlen=128, padding='post', truncating='post')

In [ ]:
y = train_data[['toxic', 'severe_toxic', 	'obscene', 	'threat', 'insult',	'identity_hate']]

**Splitting the data**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=42)

**Training Bidirectional LSTM model**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
#Creating Pytorch Data Loaders
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.long),
    torch.tensor(y_train.values, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype = torch.long),
    torch.tensor(y_test.values, dtype = torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Building Bidirectional LSTM model
class PyTorchLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(PyTorchLSTM, self).__init__()

        # 1. Embedding Layer: Converts Word IDs to dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 2. Bidirectional LSTM
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # 3. Regularisation layer to prevent overfitting
        self.dropout = nn.Dropout(0.5)

        # 4. Dense Linear Output layer
        # Hidden dim is multiplied by 2 because Bidirectional joins forward and backward outputs
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)

        # Pass through LSTM. We safely ignore the raw internal memory states (_, _)
        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Pull out only the very last word output step to represent the whole text
        last_step_out = lstm_out[:, -1, :]

        out = self.dropout(last_step_out)
        return self.fc(out)

In [ ]:
# Initializing configuration and loss
# Constants matching the current dataset setup
VOCAB_SIZE = 30000
EMBEDDING_DIM = 128
HIDDEN_DIM = 64
OUTPUT_DIM = 6 # 6 target toxicity columns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = PyTorchLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
# Training and Testing loop
EPOCHS = 4

for epoch in range(EPOCHS):
    # TRAINING PHASE
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()                  # Clear old gradients
        predictions = model(batch_x)           # Forward pass
        loss = criterion(predictions, batch_y) # Calculate error
        loss.backward()                        # Backward pass
        optimizer.step()                       # Update weights

        train_loss += loss.item()
        # Training accuracy
        probs = torch.sigmoid(predictions)
        preds_binary = (probs >= 0.5).float()
        train_correct += (preds_binary == batch_y).sum().item()
        train_total += batch_y.numel()

    # TESTING PHASE
    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0
    with torch.no_grad(): # Disable memory-heavy gradient calculations
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            test_preds = model(batch_x)
            t_loss = criterion(test_preds, batch_y)
            test_loss += t_loss.item()
            # Testing accuracy
            t_probs = torch.sigmoid(test_preds)
            t_preds_binary = (t_probs >= 0.5).float()
            test_correct += (t_preds_binary == batch_y).sum().item()
            test_total += batch_y.numel()

    epoch_train_acc = (train_correct / train_total) * 100
    epoch_val_acc = (test_correct / test_total) * 100

    print(f"Epoch {epoch+1:02} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {test_loss/len(test_loader):.4f} | Val Acc: {epoch_val_acc:.2f}%")

Epoch 01 | Train Loss: 0.1535 | Train Acc: 96.06% | Val Loss: 0.1414 | Val Acc: 96.31%
Epoch 02 | Train Loss: 0.1383 | Train Acc: 96.35% | Val Loss: 0.1192 | Val Acc: 96.31%
Epoch 03 | Train Loss: 0.0940 | Train Acc: 96.86% | Val Loss: 0.0818 | Val Acc: 96.98%
Epoch 04 | Train Loss: 0.0711 | Train Acc: 97.63% | Val Loss: 0.0683 | Val Acc: 97.77%


Model is performing very good

Testing on a new data

In [ ]:
test_data = pd.read_csv('/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/test.csv')
test_data.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [ ]:
test_data.shape

(153164, 3)

In [ ]:
test_data['cleaned_comment'] = test_data['comment_text'].apply(clean_text)
test_data.head()

,id,comment_text,cleaned_comment
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...,yo bitch ja rule succesful ever whats hating s...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...,rfc title fine imo
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...",sources zawe ashton lapland
3,00017563c3f7919a,":If you have a look back at the source, the in...",look back source information updated correct f...
4,00017695ad8997eb,I don't anonymously edit articles at all.,anonymously edit articles


In [ ]:
sequences = tokenizer.texts_to_sequences(test_data['cleaned_comment'].values)

X_test_data = pad_sequences(sequences, maxlen=128, padding='post', truncating='post')

In [ ]:
model.eval()

test_tensor = torch.tensor(X_test_data, dtype=torch.long)
test_loader = DataLoader(test_tensor, batch_size=64, shuffle=False)

# Creating a blank list to store the final probability outputs
all_predictions = []

# Generate predictions batch-by-batch to save GPU memory
with torch.no_grad():
    for batch_x in test_loader:
        batch_x = batch_x.to(device)
        raw_outputs = model(batch_x)

        # Apply Sigmoid to turn raw numbers into clean 0-1 probabilities
        probabilities = torch.sigmoid(raw_outputs)

        # Move back to CPU and convert to a NumPy array
        all_predictions.append(probabilities.cpu().numpy())

# Stack all batches together into one giant grid
final_test_preds = np.vstack(all_predictions)
print("Predictions successfully completed!")
print("Shape of the output predictions matrix:", final_test_preds.shape)

Predictions successfully completed!
Shape of your output predictions matrix: (153164, 6)


In [ ]:
# Creating a DataFrame using target columns
target_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
submission_df = pd.DataFrame(final_test_preds, columns=target_columns)
submission_df.head()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0.903557,0.280372,0.818497,0.057860,0.762129,0.169896
1,0.014012,0.000166,0.002358,0.000239,0.003434,0.000805
2,0.073209,0.000770,0.014002,0.000706,0.017509,0.002749
3,0.008167,0.000105,0.001343,0.000176,0.002026,0.000555
4,0.046162,0.000467,0.008222,0.000483,0.010682,0.001835


In [ ]:
submission_df = (submission_df >= 0.5).astype(int)

In [ ]:
submission_df.head()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,0,1,0,1,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [ ]:
submission_df.insert(0, 'id', test_data['id'].values)

In [ ]:
submission_df.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,1,0,1,0,1,0
1,0000247867823ef7,0,0,0,0,0,0
2,00013b17ad220c46,0,0,0,0,0,0
3,00017563c3f7919a,0,0,0,0,0,0
4,00017695ad8997eb,0,0,0,0,0,0


In [ ]:
final_test_data = pd.merge(test_data, submission_df, on = 'id', how = 'inner')
final_test_data.head()

,id,comment_text,cleaned_comment,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...,yo bitch ja rule succesful ever whats hating s...,1,0,1,0,1,0
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...,rfc title fine imo,0,0,0,0,0,0
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...",sources zawe ashton lapland,0,0,0,0,0,0
3,00017563c3f7919a,":If you have a look back at the source, the in...",look back source information updated correct f...,0,0,0,0,0,0
4,00017695ad8997eb,I don't anonymously edit articles at all.,anonymously edit articles,0,0,0,0,0,0


In [ ]:
final_test_data.drop('cleaned_comment', axis=1, inplace=True)

In [ ]:
final_test_data.to_csv('/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/FinalTestData.csv')

**Saving the Pytorch Model**

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/pytorch_toxicity_lstm.pt')
print("PyTorch weights successfully stored for deployment!")


PyTorch weights successfully stored for deployment!


**Saving tokenizer**

In [ ]:
import pickle
with open("/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/keras_tokenizer.pkl", 'wb') as f:
  pickle.dump(tokenizer, f)